In [3]:
# ============================================================
# WORD EMBEDDING
# Word Vectorization using Word2Vec and GloVe
# ============================================================

# Install if needed:
# pip install numpy==1.26.4 scipy==1.12.0 gensim==4.3.2 nltk scikit-learn

import nltk
import numpy as np
from nltk.tokenize import word_tokenize
from gensim.models import Word2Vec
from numpy.linalg import norm

nltk.download("punkt")

# ============================================================
# 1. SAMPLE CORPUS
# ============================================================

sentences = [
    "I love natural language processing",
    "I love machine learning",
    "I enjoy deep learning",
    "Natural language processing is part of artificial intelligence",
    "Machine learning is used in artificial intelligence",
    "Deep learning is a branch of machine learning",
    "Word embedding represents words as vectors",
    "GloVe and Word2Vec are popular word embedding techniques"
]

tokenized_sentences = [
    word_tokenize(sentence.lower())
    for sentence in sentences
]

print("Tokenized Sentences:")
print(tokenized_sentences)


# ============================================================
# 2. WORD2VEC
# ============================================================

word2vec_model = Word2Vec(
    sentences=tokenized_sentences,
    vector_size=100,
    window=3,
    min_count=1,
    workers=4,
    sg=1  # sg=1 Skip-Gram, sg=0 CBOW
)

print("\n=== Word2Vec Vector Example ===")
print("Vector for 'learning':")
print(word2vec_model.wv["learning"])

print("\nVector Dimension:")
print(len(word2vec_model.wv["learning"]))

print("\nMost Similar Words to 'learning':")
print(word2vec_model.wv.most_similar("learning", topn=5))

print("\nSimilarity between 'machine' and 'learning':")
print(word2vec_model.wv.similarity("machine", "learning"))


# ============================================================
# 3. SAVE AND LOAD WORD2VEC MODEL
# ============================================================

word2vec_model.save("word2vec_model.model")

loaded_word2vec = Word2Vec.load("word2vec_model.model")

print("\nLoaded Word2Vec Model Test:")
print(loaded_word2vec.wv.most_similar("learning", topn=3))


# ============================================================
# 4. SENTENCE VECTOR USING WORD2VEC
# ============================================================

def sentence_vector_word2vec(sentence, model):
    tokens = word_tokenize(sentence.lower())
    vectors = []

    for token in tokens:
        if token in model.wv:
            vectors.append(model.wv[token])

    if len(vectors) == 0:
        return np.zeros(model.vector_size)

    return np.mean(vectors, axis=0)


sentence = "I love machine learning"

sentence_vec = sentence_vector_word2vec(sentence, word2vec_model)

print("\nSentence Vector using Word2Vec:")
print(sentence_vec)
print("Sentence Vector Dimension:", len(sentence_vec))


# ============================================================
# 5. GLOVE
# ============================================================
# Download GloVe first:
# https://nlp.stanford.edu/projects/glove/
#
# Example file:
# glove.6B.50d.txt
#
# Put the file in the same folder as this Python file.

def load_glove_model(file_path):
    glove_model = {}

    with open(file_path, "r", encoding="utf-8") as file:
        for line in file:
            values = line.split()
            word = values[0]
            vector = np.asarray(values[1:], dtype="float32")
            glove_model[word] = vector

    return glove_model


# Change this path according to your file location
glove_file_path = "./Glove/glove.6B.50d.txt"

try:
    glove_model = load_glove_model(glove_file_path)

    print("\n=== GloVe Loaded Successfully ===")
    print("Total Words:", len(glove_model))

    print("\nGloVe Vector for 'king':")
    print(glove_model["king"])

    print("\nVector Dimension:")
    print(len(glove_model["king"]))

except FileNotFoundError:
    glove_model = None
    print("\nGloVe file not found.")
    print("Please download glove.6B.50d.txt and place it in the same folder.")


# ============================================================
# 6. COSINE SIMILARITY FOR GLOVE
# ============================================================

def cosine_similarity(vec1, vec2):
    return np.dot(vec1, vec2) / (norm(vec1) * norm(vec2))


if glove_model is not None:
    word1 = "king"
    word2 = "queen"

    similarity = cosine_similarity(glove_model[word1], glove_model[word2])

    print(f"\nGloVe Similarity between '{word1}' and '{word2}':")
    print(similarity)


# ============================================================
# 7. MOST SIMILAR WORDS USING GLOVE
# ============================================================

def most_similar_glove(word, glove_model, top_n=5):
    if word not in glove_model:
        return f"'{word}' not found in vocabulary."

    target_vector = glove_model[word]
    similarities = {}

    for other_word, other_vector in glove_model.items():
        if other_word != word:
            similarities[other_word] = cosine_similarity(target_vector, other_vector)

    sorted_words = sorted(
        similarities.items(),
        key=lambda x: x[1],
        reverse=True
    )

    return sorted_words[:top_n]


if glove_model is not None:
    print("\nMost Similar Words to 'king' using GloVe:")
    results = most_similar_glove("king", glove_model, top_n=5)

    for word, score in results:
        print(word, score)


# ============================================================
# 8. WORD ANALOGY USING GLOVE
# Example: king - man + woman ≈ queen
# ============================================================

def glove_analogy(word_a, word_b, word_c, glove_model, top_n=5):
    if word_a not in glove_model or word_b not in glove_model or word_c not in glove_model:
        return "One or more words not found in vocabulary."

    target_vector = glove_model[word_a] - glove_model[word_b] + glove_model[word_c]

    similarities = {}

    for word, vector in glove_model.items():
        if word not in [word_a, word_b, word_c]:
            similarities[word] = cosine_similarity(target_vector, vector)

    sorted_words = sorted(
        similarities.items(),
        key=lambda x: x[1],
        reverse=True
    )

    return sorted_words[:top_n]


if glove_model is not None:
    print("\nGloVe Analogy: king - man + woman ≈ ?")
    analogy_results = glove_analogy("king", "man", "woman", glove_model)

    for word, score in analogy_results:
        print(word, score)


# ============================================================
# 9. SENTENCE VECTOR USING GLOVE
# ============================================================

def sentence_vector_glove(sentence, glove_model, vector_size=50):
    tokens = word_tokenize(sentence.lower())
    vectors = []

    for token in tokens:
        if token in glove_model:
            vectors.append(glove_model[token])

    if len(vectors) == 0:
        return np.zeros(vector_size)

    return np.mean(vectors, axis=0)


if glove_model is not None:
    sentence = "I love machine learning"

    glove_sentence_vec = sentence_vector_glove(
        sentence,
        glove_model,
        vector_size=50
    )

    print("\nSentence Vector using GloVe:")
    print(glove_sentence_vec)
    print("Sentence Vector Dimension:", len(glove_sentence_vec))

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\ASUS\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


Tokenized Sentences:
[['i', 'love', 'natural', 'language', 'processing'], ['i', 'love', 'machine', 'learning'], ['i', 'enjoy', 'deep', 'learning'], ['natural', 'language', 'processing', 'is', 'part', 'of', 'artificial', 'intelligence'], ['machine', 'learning', 'is', 'used', 'in', 'artificial', 'intelligence'], ['deep', 'learning', 'is', 'a', 'branch', 'of', 'machine', 'learning'], ['word', 'embedding', 'represents', 'words', 'as', 'vectors'], ['glove', 'and', 'word2vec', 'are', 'popular', 'word', 'embedding', 'techniques']]

=== Word2Vec Vector Example ===
Vector for 'learning':
[-5.1986083e-04  2.4410448e-04  5.0848415e-03  9.0568047e-03
 -9.3175778e-03 -7.1077077e-03  6.4730560e-03  8.9935092e-03
 -5.0347368e-03 -3.7738127e-03  7.3997704e-03 -1.5469088e-03
 -4.4945576e-03  6.5816585e-03 -4.8531969e-03 -1.8244253e-03
  2.9200246e-03  9.9694135e-04 -8.3235782e-03 -9.4956718e-03
  7.3031005e-03  5.0847461e-03  6.7345174e-03  7.5738429e-04
  6.3724746e-03 -3.4265087e-03 -9.5063628e-04  5